<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Lesson 4 精华 — Agents with Memory

**一句话定位**:**HITL 反馈不再白费** —— 用 LangGraph Store 把每次反馈 **持久化** 成 profile,下次自动用上;assistant 越用越懂你。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🤔 学之前先想清楚:为什么 ambient agent 必须有 Memory?**

**HITL 是「**安全带**」**,Memory 是「**经验积累**」**。两者结合才完整:

| 没 Memory | 有 Memory |
|----------|----------|
| 每次都要 **重复纠正同样的错** | 系统 **记住偏好**,下次自动正确 |
| 用户越用越烦(要一直手动改) | **越用越省心**(干预越来越少) |
| assistant **永远 38 岁** | assistant **会成长** |

→ Memory 是 ambient agent 「**值得长期使用**」的前提。

本节学会的话,你的 assistant 就有了「**学习能力**」—— 不是模型层面的 fine-tune,而是 **应用层的渐进式适应**,成本极低、效果立竿见影。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 LangGraph 的两种 Memory — 先建心智模型

</div>

LangGraph 里有 **两种 memory**,它们 **互补**,千万别搞混:

| 维度 | 🧵 **Thread-Scoped(短期)** | 🌐 **Across-Thread(长期)** |
|------|--------------------------|---------------------------|
| **范围** | 单次会话内 | **跨多次会话** |
| **载体** | `checkpointer`(同 thread_id 之间共享) | **`Store`**(全局,跨 thread_id) |
| **存什么** | 当次对话历史、临时变量 | **用户偏好、长期事实、累积知识** |
| **类比** | RAM(工作内存,断电即失) | **硬盘**(持久化,跨进程) |
| **API 操作** | 自动(节点 return dict 就更新 state)| **手动**(`store.put` / `store.get` / `store.search`)|
| **前面哪节用过** | Lesson 0 / 3(HITL 续跑就靠它) | **本节新登场** |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 关键理解:同一个 graph 可以**同时**用两种 memory**

**编译时同时挂上**:

```python
graph.compile(
    checkpointer=InMemorySaver(),   # ← thread-scoped(每会话)
    store=InMemoryStore(),          # ← across-thread(跨会话)
)
```

节点函数里就能 **同时访问** 当次对话(`state["messages"]`)和长期偏好(`store.get(namespace, key)`)。

**这是 ambient agent 设计的标准范式**:短期 state 跑流程,长期 store 存偏好。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🗄️ LangGraph Store — 三种实现

</div>

Store 提供 **统一 API**,底层可以是 3 种实现,**根据部署场景选**:

| 实现 | 持久化 | 速度 | 用途 | 怎么得到 |
|------|--------|------|------|---------|
| **`InMemoryStore`** | ❌ 进程结束丢光 | ⚡ 最快 | Notebook 实验、单元测试 | `from langgraph.store.memory import InMemoryStore` |
| **`langgraph dev`** 内置 | ⚠️ pickle 到本地文件 | ⚡ 快 | 开发期 + Studio 调试 | `langgraph dev` 自动注入 |
| **PostgreSQL + pgvector** | ✅ 完全持久 | 🐢 较慢但稳 | **生产** | LangGraph Platform Cloud SaaS 自动配 |

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**💡 学习要点:API 统一 = 部署无痛切换**

**不管底层是哪种**,你的 agent 代码用的都是同样的 `store.put / store.get / store.search`。

→ 开发用 `InMemoryStore`,生产用 Postgres,**业务代码一字不改**。这是 LangGraph 「**层级抽象**」哲学的体现 —— 跟 SQLAlchemy 让你换 DB 引擎不用改代码是一个道理。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📦 Store 的 4 个核心操作

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# 1️⃣ 写入 ── store.put(namespace, key, value)
<span style="background:#3d3414; color:#fde047; padding:0 2px;">namespace = ("user_42", "calendar_prefs")</span>   # ← tuple,可层级
store.put(namespace, key="morning_pref", value={"text": "喜欢上午开会"})

# 2️⃣ 单条读取 ── store.get(namespace, key)
item = store.get(namespace, key="morning_pref")
print(item.value)  # {'text': '喜欢上午开会'}

# 3️⃣ 列出 namespace 下所有 ── store.search(namespace)
items = store.search(namespace)   # 返回 list,最新的在最后
for it in items:
    print(it.key, it.value)

# 4️⃣ 删除 ── store.delete(namespace, key)
store.delete(namespace, key="morning_pref")
</code></pre>

### 🏷️ Namespace 是 tuple,这点很重要

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 为什么用 tuple 而不是字符串?**

tuple 像 **文件夹路径**,天然支持 **层级**:

```python
("user_42",)                          # 用户 42 的所有 memory
("user_42", "calendar_prefs")         # 仅日历偏好
("user_42", "calendar_prefs", "work") # 工作日历偏好
```

**好处**:

- `store.search(("user_42",))` 返回 **整个用户** 的所有 memory(不管哪一层)
- `store.search(("user_42", "calendar_prefs"))` 只返回 **日历相关**
- 多用户场景下,**namespace 第一层用 user_id**,数据天然隔离

**学生记忆口诀**:**「**Store 是带文件夹的 Redis**」**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔄 HITL 反馈 → Memory 更新 — 整套机制

</div>

这是本节的 **核心机制**,理解了这套流程,memory 就 「**通了**」:

```
用户在 Agent Inbox 上做操作(edit / response / ignore)
         │
         ▼
interrupt_handler 检测到反馈类型
         │
         ▼
调用 update_memory(store, namespace, messages)
         │
         ▼
┌────────────────────────────────┐
│ ① 从 store 读出 现有 profile (str) │
│                                │
│ ② 把 现有profile + 用户反馈 + ctx  │
│   塞进一个特定 prompt          │
│                                │
│ ③ LLM 跑一遍,产出 新profile     │
│   (with_structured_output:     │
│   { new_profile, justification})│
│                                │
│ ④ store.put(namespace, profile) │
│   覆盖写入新版                  │
└────────────────────────────────┘
         │
         ▼
下次 LLM 调用时,把新 profile 注入 system prompt → 自动用上
```

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 整套机制最核心的一句话**

**Memory 系统 = 「**LLM 自己做 diff & merge**」**。

用户给的反馈是 **自然语言**(「**短一点,正式点**」),不是结构化数据。我们 **让 LLM** 看 现有profile + 新反馈,产出 **整合后的 profile** —— 不需要传统数据库的 schema 设计、字段迁移、版本管理。

→ **传统软件靠 schema 演化处理「学习」,memory 系统靠 LLM 演化**。这是 2026 年应用层 AI 的标志性范式。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📋 3 个 Memory Profile — 本课怎么划分的

</div>

本课的 email assistant 把偏好分成 **3 个独立的 profile**,各存各的 namespace:

| Profile | namespace | 装什么 | 在哪里被「**读**」 | 在哪里被「**写**」 |
|---------|-----------|--------|------------------|------------------|
| **`triage_preferences`** | `("email_assistant", "triage_preferences")` | 哪些邮件该回 / 该忽略 / 该提醒 | `triage_router` 的 prompt | 用户 ignore / response 一封 notify 邮件时 |
| **`cal_preferences`** | `("email_assistant", "cal_preferences")` | 会议偏好(时长、时段、命名风格) | `llm_call` 的 prompt | 用户 edit / response `schedule_meeting` 时 |
| **`response_preferences`** | `("email_assistant", "response_preferences")` | 邮件回复风格(简洁 / 正式 / 结尾习惯) | `llm_call` 的 prompt | 用户 edit / response `write_email` 时 |

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**💡 为什么分 3 个,不是 1 个大 profile?**

理由 = **关注点分离**:

- 用户改邮件正文,**只影响** `response_preferences`(不动 cal)
- 用户改会议时长,**只影响** `cal_preferences`(不动 response)
- LLM 改写时,**输入更聚焦** → 不容易混淆 → 更新更精准

**反例**:如果是 1 个大 profile,LLM 看 5KB 文字写一句新的「**用户喜欢简短回信**」,**99% 概率会顺手改坏其他无关偏好**(LLM 注意力是有限的)。

→ **profile 设计 = 上下文工程**。小而专的 profile > 大而全。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ `update_memory` 函数 — 真正的代码

</div>

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">from pydantic import BaseModel, Field

# 1️⃣ Schema —— LLM 必须按这个结构输出
class UpdateProfile(BaseModel):
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">new_profile: str</span>      = Field(description="完整的新 profile,包含现有内容 + 新偏好")
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">justification: str</span>    = Field(description="为什么这样改 / 提取了哪些新偏好")

# 2️⃣ Update 函数 —— 核心逻辑
def update_memory(store, namespace, messages):
    # 读出现有 profile(不存在则用 default)
    item = store.get(namespace, key="profile")
    current = item.value["text"] if item else DEFAULT_PROFILE

    # 让 LLM 写新版
    llm_with_schema = llm.<span style="background:#3d3414; color:#fde047; padding:0 2px;">with_structured_output(UpdateProfile)</span>
    result = llm_with_schema.invoke([
        {"role": "system", "content": MEMORY_UPDATE_PROMPT},
        {"role": "user", "content":
            f"现有 profile:\n{current}\n\n"
            f"用户最新反馈和上下文:\n{messages}\n\n"
            f"请输出整合后的新 profile,以及 justification(说明改了哪里、为什么)"
        }
    ])

    # 写回 store
    store.put(namespace, key="profile", value={"text": result.new_profile})
    return result.justification   # 返回理由,可以打日志、给 LangSmith
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 3 个值得注意的设计**

1. **`with_structured_output`** —— 强制 LLM 返回符合 schema 的对象,而不是裸文本。**避免解析失败**。

2. **`justification` 字段** —— 不只是为了好看,这是 **可解释性** 的关键。LangSmith trace 里你能看到「**LLM 为什么这样改 memory**」,出问题时能 debug。

3. **prompt 里同时给「**现有 profile**」+「**新反馈**」+「**让 LLM 整合**」的指令** —— 这是经典的 **「**LLM 做合并**」** 模式。GPT-4.1 / Claude Sonnet 4.5 在这种任务上很可靠。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📊 3 种 HITL 反馈 → 学到什么 memory?

</div>

本课最有趣的实验:**同一封邮件,3 种 HITL 反馈方式,memory 学到的东西完全不同**。

| 用户操作 | 学习信号强度 | Memory 变化 | 为什么? |
|---------|-------------|------------|---------|
| **`accept`** | ❌ **无** | **不变** | 「同意」 ≠ 「告诉系统改什么」 —— 没对比 = 没信号 |
| **`edit`** | ⭐⭐⭐ **最强** | **精确学习**:LLM 看 原版 vs 修改版 的 diff,**提取偏好** | 「**改成 30 分钟**」是 **明确的、具体的、可泛化的**信号 |
| **`response`** | ⭐⭐ **中等** | **从自然语言提取**:LLM 翻译「**短一点**」成「**偏好简洁邮件**」 | 比 edit 模糊,但比 accept 信息量大得多 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 学生最容易踩坑的认知:「**Accept 就是同意,系统应该学习啊?**」**

**不学**。原因:**Accept 既可能意味着「**完美**」,也可能意味着「**算了懒得改**」**。系统无法区分这两种情况 → **没有学习信号**。

→ **想让 agent 学,必须给「**有方向**」的反馈**(edit 或 response)。这也告诉我们:**做 ambient agent 时,Agent Inbox UI 设计要鼓励用户用 edit/response,而不是无脑 accept**(否则系统永远学不到东西)。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🌟 Memory 进阶 — 2026 视角

</div>

本课为了清晰,**用了最简单的 memory schema**:

- profile = **一个字符串**
- 更新 = **整个字符串覆盖**

这够入门,**生产场景需要升级**。3 个常见进阶方向:

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 进阶 1:语义搜索(Semantic Search)**

当 memory 变多(几百几千条),**遍历找相关的偏好太慢**。**LangGraph Store 内置 pgvector 支持**:

```python
store.put(namespace, key, value, index=["embeddings"])
results = store.search(namespace, query="how does user prefer meetings")
# 返回按相似度排序的 top-k
```

→ 适合 **大量碎片化 memory**(每条都是一小段事实)的场景。

</div>

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 进阶 2:结构化 schema(不再用字符串)**

用 Pydantic class 存 memory,字段化:

```python
class CalendarPrefs(BaseModel):
    preferred_duration_min: int = 30
    preferred_time_window: tuple[int, int] = (14, 18)  # 下午 2-6 点
    avoid_days: list[str] = []
```

→ 更精确、可查询、可分析,但 **更新逻辑更复杂**(需要 schema migration)。

</div>

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 进阶 3:LangMem —— 生产级 memory 管理库**

2025 年下半年 LangChain 团队推出 [**LangMem**](https://langchain-ai.github.io/langmem/),专门做 memory 管理。封装了:

| 能力 | 说明 |
|------|------|
| **多种 memory 类型** | facts / events / preferences / reflections |
| **自动合并去重** | 不会越存越多重复内容 |
| **TTL / 衰减** | 旧 memory 自动失效 |
| **跨 thread 检索 + 排序** | 找相关 memory 用语义+元数据双路 |
| **可演化 schema** | 字段可加可减,不破坏老数据 |

**结论**:**生产用 memory,别造轮子,直接上 LangMem**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎭 类比:Memory 像什么?

</div>

### 🅰️ 类比 1:跟同事磨合

新同事第一次帮你订机票,可能订错时段、选错座位。你给反馈:「**靠窗,早班机**」。**TA 记住后,下次自动这样选**。

Memory 就是 agent 的「**长期工作笔记**」 —— 它替你记住,你不用每次重复。

### 🅱️ 类比 2:浏览器记住偏好

你登录某网站,改了语言、字号、夜间模式。**网站记住** —— 下次打开自动这样。

差别:网站记的是 **配置**(明确字段),agent 记的是 **意图**(自然语言 profile)。本质都是 **「**让系统适应你**」**。

### 🅲 类比 3:Recommender 系统

Netflix 看一段时间后,推荐越来越准。它在背后维护一个 **「**你的口味模型**」**,每次你看 / 评分都在更新它。

**Memory 跟 recommender 的同与异**:

| 维度 | Recommender | Agent Memory |
|------|-------------|--------------|
| 信号 | 看 / 跳过 / 评分 | edit / response / ignore |
| 更新方式 | 数值 weight 增减(传统 ML)| **LLM 自然语言整合** |
| 可解释 | 难(向量算的) | **直接读字符串就懂** |
| 修改 | 难(需要重训) | **直接编辑 profile 文本就行** |

→ **Agent Memory 在「**可解释性**」和「**可控性**」上完胜传统 recommender**。这是 LLM 时代应用层的优势。

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 6 个最容易踩的坑**

1. **❌ 忘了在 `.compile()` 里挂 store** → 所有 `store` 操作都不报错但没效果(节点拿到的是不挂 graph 的临时 store)。

2. **❌ namespace 写成字符串** → `store.put("calendar_prefs", ...)` 报错。**必须是 tuple**,即使只有一层也要 `("calendar_prefs",)`(注意逗号)。

3. **❌ 让 LLM 自由生成 new_profile,没用 `with_structured_output`** → 解析失败、JSON 错乱、profile 被破坏。**永远用 structured output**。

4. **❌ 所有偏好塞 1 个 profile** → LLM 改一处坏一处。**按用途分多个 profile**。

5. **❌ 每次操作都更新 memory(包括 accept)** → LLM 看到「accept」也尝试提炼偏好 → 产出垃圾。**只在 edit / response 时才更新**。

6. **❌ 更新 prompt 没要求「保留现有内容」** → LLM 可能把旧 profile 整个扔了,只写新偏好。**prompt 必须说**「整合现有 + 新偏好,**不要删除已有信息**」**。

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**Memory 的核心 = HITL 反馈 → LLM 整合成 profile → 写 Store → 下次 LLM 调用前注入 prompt → 行为对齐用户偏好**。

### 🎯 学完这节,你应该能回答:

1. ✅ **Thread-Scoped 和 Across-Thread Memory 怎么区分?**(RAM vs 硬盘)
2. ✅ **Store 的 3 种实现什么时候用哪个?**(InMemory 实验 / langgraph dev 本地 / Postgres 生产)
3. ✅ **为什么 namespace 用 tuple?**(支持层级、像文件夹路径)
4. ✅ **Memory 更新的核心机制是什么?**(LLM 看现有 + 反馈,自己 diff & merge)
5. ✅ **为什么 Accept 不触发学习?**(没对比 = 没信号)
6. ✅ **为什么用多个小 profile 而不是一个大 profile?**(关注点分离,LLM 注意力有限)

### 🎁 整门课走完了!

**完整闭环**:Build → Eval → HITL → **Memory**。

| 模块 | 给 agent 加了什么能力 |
|------|---------------------|
| **M2 Build** | 能 **跑** |
| **M3 Eval** | 能 **测** |
| **M4 HITL** | 能 **被信任部署** |
| **M5 Memory** | 能 **越用越好** |

🚀 **下一步可以做的事**:

- 把这套架构 **迁移** 到你的真实场景(Slack 监控、Linear 自动分类、CI 通知处理...)
- 加 **LangMem** 替换简单 string profile
- 用 **LangGraph Platform** 部署到生产,接你的真实 Gmail / Slack

📎 配套阅读:[`4_memory.ipynb` 主课](./4_memory.ipynb) · [`3_z_⭐️_本节精华_HITL.ipynb`](./3_z_⭐️_本节精华_HITL.ipynb) · [`_课程总览_🏞️.svg`](../_课程总览_🏞️.svg)

</div>